# Quickstart Guide

The following is an introduction to SciBmad in one copy-and-pastable block: build a FODO cell, compute its linear, nonlinear, chromatic and spin optics, adjust the sextupoles to set the chromaticities, and track particles with spin. The rest of this page walks through it step by step. To easily copy a block of code anywhere in the documentation, move your cursor to the top right of the code block and select to copy.

In [1]:
# Hidden setup: keep printed tables narrow enough for the page.
import scibmad as sb
sb.options.display_width = 100
sb.options.display_height = 30

## A Simple Copy-Paste Example


```python
import re
import numpy as np
import scibmad as sb

# Define lattice elements, named after their variables
with sb.elements():
    qf = sb.Quadrupole(Kn1=0.36, L=0.5)
    sf = sb.Sextupole(Kn2=1.2, L=0.2)
    d = sb.Drift(L=0.1)
    b = sb.SBend(L=1.2, angle=np.pi/132)
    qd = sb.Quadrupole(Kn1=-0.36, L=0.5)
    sd = sb.Sextupole(Kn2=-1.2, L=0.2)

# Construct a Beamline and define the reference species + momentum
fodo = sb.Beamline([qf, sf, b, d, qd, sd, b, d],
                   species_ref=sb.Species("electron"), pc_ref=18e9)

# Change the quadrupole strengths
qf.Kn1 = 0.37
qd.Kn1 = -0.37

# Set number of integration steps in qd to 3
qd.n_steps = 3

# Find the single instance of `qf` in fodo:
qf_in_fodo = fodo[qf]

# Find the drifts `d` in `fodo`
ds = fodo[[d]]

# Get the first element of `fodo`
first_ele = fodo[1]

# Gets all quadrupoles
quads = fodo[lambda x: x.kind == "Quadrupole"]  # Lambda function
quads = fodo["Quadrupole::*"]                   # Classic Bmad syntax
quads = fodo[re.compile("q.*")]                 # Regex match on name

# Compute the periodic optics
tw = sb.twiss(fodo)

# Print the tunes, phase slip factor (etac), momentum compaction (alphac)
print("tw.q1 =", tw.q1)          # Horizontal-like tune
print("tw.q2 =", tw.q2)          # Vertical-like tune
print("tw.etac =", tw.etac)      # Phase slip factor
print("tw.alphac =", tw.alphac)  # Momentum compaction

# Print the horizontal-like beta function at various locations
# including inside elements
print(tw.beta1[qd])          # Beginning of element qd
print(tw.beta1[qd, "exit"])  # End of qd
print(tw.beta1[qd, :])       # Every integration step inside qd
print(tw.beta1[qd, 2])       # 2nd integration step inside qd
print(tw.beta1[4, :])        # Every integration step of element 4
print(tw.beta1[0:1.23])      # Every integration step between s=0, s=1.23

# Compute only specific quantities
#   "beta1" = horizontal-like beta
#   "phi2" = vertical-like phase advance,
#   "dx" = horizontal dispersion
tw = sb.twiss(fodo, cols=["beta1", "phi2", "dx"])

# Compute higher order chromatic quantities such as
# the chromaticity, Montague W functions ("w1", "w2"),
# and 2nd order dispersions ("dx_2", "dy_2") by setting
# the order of δ (chrom) equal to 2
tw = sb.twiss(fodo, cols=["w1", "w2", "dx_2", "dy_2"], chrom=2)
print(repr(tw.q2))
# Prints:
#   AmplitudeDependentValue:
#    0.05401330001000863 - 0.008253153240321323 δ

# Get the y-chromaticity
chromy = tw.q2(delta=1)
chromy = tw.dq2  # or equivalently tw.dq2_1

# Resonance driving terms h3000, h2100
tw = sb.twiss(fodo, cols=["h3000", "h2100"])

# Spin analysis (invariant spin field as Taylor series,
# amplitude-dependent spin tune).
tw = sb.twiss(fodo, cols=["nx", "ny", "nz"], spin=True,
              as_taylor_series=True, order=2)
print(repr(tw.qspin))
# Prints:
#   AmplitudeDependentValue:
#    -0.3094612779434053 - 0.3094612776559473 δ - 4.0543183384536337e-10 J₁
#    + 28.127501690102246 J₂ - 2.950933476050364e-10 δ²

# Adjust the sextupoles to make chromaticities to be +1
# using SciBmad's autodiff + parametric normal form
te = sb.TwissExpr(fodo)
sol = sb.match(vary=[sb.Knob(sf, "Kn2"), sb.Knob(sd, "Kn2")],
               targets=[te.dq1 == 1, te.dq2 == 1])

# Track a particle with [x, px, y, py, z, pz]
# = [1e-3, 0, 0, 0, 0, 1e-3] for n_turns=10
res = sb.track(fodo, v0=[[1e-3, 0, 0, 0, 0, 1e-3]], n_turns=10)

# Also do spin quaternion tracking:
res = sb.track(fodo, v0=[[1e-3, 0, 0, 0, 0, 1e-3]], n_turns=10, spin=True)
# Apply spin quaternion to horizontal initial spin direction (FAST):
s = sb.track_spin(res.q, [1, 0, 0])

# Track a bunch, give an array of size n_particles x 6:
n_particles = 10
res = sb.track(fodo, v0=np.random.rand(n_particles, 6)*1e-5, n_turns=10, spin=True)
```

:::{only} python
SciBmad is a Julia package, and the Python package `scibmad` runs it: every name of SciBmad is
a name of `scibmad`, with the same arguments. How Python writes the few things that Julia writes
differently is described in [SciBmad from Python](python.md).
:::

## Step-By-Step Description

### Constructing a Beamline

Let's start by constructing a simple FODO cell `Beamline`: a focusing quadrupole ("F"), a
gap ("O"), a defocusing quadrupole ("D") and another gap, the basic repeating cell of many
rings. Here the gaps hold a sextupole, a bending magnet and a drift.

To do this, we will define six `LatElement`s, one for each kind of object in the cell. The
lengths of each object are specified by `L` (in meters), and the quadrupole strengths can be
set using the property `Kn1`, where `n` means the "normal" multipole (`s` would be "skew")
and `1` means 1st order multipole (quadrupole). The normal sextupole strength is thus `Kn2`.
These strengths are normalized by the magnetic rigidity of the reference particle: `Kn1` is in
1/m² and `Kn2` in 1/m³.
For the bend, the `angle` parameter sets both the curvature of the coordinate system and the
corresponding magnetic field so that a "reference particle" will follow the coordinate
system curvature exactly. The coordinate system curvature and normalized field strength can
be set independently using `g_ref` and `Kn0` respectively if desired.

In [2]:
import re
import numpy as np
import scibmad as sb

# Define lattice elements, named after their variables
with sb.elements():
    qf = sb.Quadrupole(Kn1=0.36, L=0.5)
    sf = sb.Sextupole(Kn2=1.2, L=0.2)
    d = sb.Drift(L=0.1)
    b = sb.SBend(L=1.2, angle=np.pi/132)
    qd = sb.Quadrupole(Kn1=-0.36, L=0.5)
    sd = sb.Sextupole(Kn2=-1.2, L=0.2)

# Construct a Beamline and define the reference species + momentum
fodo = sb.Beamline([qf, sf, b, d, qd, sd, b, d],
                   species_ref=sb.Species("electron"), pc_ref=18e9)

:::{only} julia
Wrapping the definitions in `@elements` sets the `name` of each element to the name of its
variable (`qf.name` is `"qf"`).
:::
:::{only} python
Defining the elements in a `with sb.elements():` block sets the `name` of each element to the
name of its variable (`qf.name` is `"qf"`).
:::

The `Beamline` is designed for a reference particle, here an
electron with the momentum `pc_ref` = 18 GeV/c.

All element "kinds" (`Sextupole`, `Quadrupole`, etc.) are one single type `LatElement` 
under the hood. Therefore, there is nothing preventing you from writing e.g.
`Drift(L=1.2, Ks8=123)`. This structure provides maximal flexibility for defining and
modifying elements (see [Defining a LatElement](element.md) for more). A default tracking 
method is chosen given an element kind, e.g. `MatrixKick` for a finite-length `Quadrupole`, 
which you can change at any time (see [Tracking Methods](tracking-methods.md)).

`LatElement`s in a beamline simply inherit all properties from the original
element definition. For example, retuning the quadrupole strengths or number of integration 
steps

In [3]:
# Change the quadrupole strengths
qf.Kn1 = 0.37
qd.Kn1 = -0.37

# Set number of integration steps in qd to 3
qd.n_steps = 3

is reflected everywhere that element has been placed in a `Beamline`, without needing to
rebuild `fodo` or search-and-replace through its element list (see
[Defining a Beamline](beamline.md) for more). 

The [`AtomicAndPhysicalConstants`](https://github.com/bmad-sim/AtomicAndPhysicalConstants.jl)
package is used for specifying particle species, and so any species defined by that package
may be provided. Also, instead of specifying the reference momentum `pc_ref` (in eV/c), we
alternatively could have specified the total reference energy `E_ref` (in eV), or the _signed_
magnetic rigidity `p_over_q_ref` (in T·m).

We can find instances of elements in a `Beamline`, by searching in the following manners:

In [4]:
# Find the *single* instance of `qf` in fodo:
qf_in_fodo = fodo[qf]

# Find *all* the drifts `d` in `fodo` (note the extra [])
ds = fodo[[d]]

# Get the first element of `fodo`
first_ele = fodo[1]

# Gets all quadrupoles
quads = fodo[lambda x: x.kind == "Quadrupole"]  # Lambda function
quads = fodo["Quadrupole::*"]                   # Classic Bmad syntax
quads = fodo[re.compile("q.*")]                 # Regex match on name

:::{only} python
`fodo[1]` is the first element: the elements of a beamline are numbered from 1, as in Julia
(see [SciBmad from Python](python.md)).
:::

To see all the ways you can search for elements, see [Defining a Beamline](beamline.md).

### Twiss

Once a `Beamline` is defined, we can compute its optics with `twiss`:

In [5]:
# Compute the periodic optics
tw = sb.twiss(fodo)

# Compute the optics of fodo as an open line (not a ring), from given
# initial Twiss parameters at its beginning
two = sb.twiss(fodo, initial=dict(beta1=13, alpha1=-1, beta2=10, alpha2=1))

By default this computes the closed orbit, tunes/phase advances and, in the Sagan-Rubin/Edwards-Teng coupling formalism, the beta functions, alpha functions, coupling matrix, phase advances/slip, and (crab) dispersions at _all integration steps_. The table itself is `tw.df`, a [`DataFrame`](https://dataframes.juliadata.org/stable/).
The numbers `1` and `2` in names such as `beta1` and `q1` refer to the two transverse normal
modes, which are horizontal-like and vertical-like (`x` and `y` when there is no coupling).

The values for the whole ring, such as the tunes, are properties of `tw`:

In [6]:
print("tw.q1 =", tw.q1)          # Horizontal-like tune
print("tw.q2 =", tw.q2)          # Vertical-like tune
print("tw.etac =", tw.etac)      # Phase slip factor
print("tw.alphac =", tw.alphac)  # Momentum compaction

tw.q1 = 0.05489859401259565
tw.q2 = 0.05401330001000861
tw.etac = 0.019165878242797245
tw.alphac = 0.019165879048722943


:::{only} julia
In Julia, the `@` sign specifies a macro, which modifies the code it acts on. In this case, `@show tw.q1` is syntactic sugar that replaces the line of code with `println("tw.q1 = ", tw.q1)`.
:::

The columns of the Twiss table contain the s-dependent values, and can be read at any location, for example the beginning/ends of elements, or every integration step over an s-range.

In [7]:
# Betas at various locations:
print(tw.beta1[qd])          # Beginning of element qd
print(tw.beta1[qd, "exit"])  # End of qd
print(tw.beta1[qd, :])       # Every integration step inside qd
print(tw.beta1[qd, 2])       # 2nd integration step inside qd
print(tw.beta1[4, :])        # Every integration step of element 4
print(tw.beta1[0:1.23])      # Every integration step between s=0, s=1.23

10.129870326396501
10.129754513094344
[10.12987033  9.91906626  9.91902817 10.12975451]
9.91902817498379
[10.32276754 10.12987033]
[13.41064349 13.41048035 12.92522013]


[10.12987033  9.91906626  9.91902817 10.12975451]
9.91902817498379
[10.32276754 10.12987033]
[13.41064349 13.41048035 12.92522013]


:::{only} python
An s-range is a slice of floats (`0:1.23`, in Julia the tuple `(0, 1.23)`).
:::
All of the ways to specify a location are described in [Twiss](twiss.md).

If only a subset of quantities is needed, the `cols` keyword restricts the computation (and
the columns returned) to exactly what you ask for, which can save computation time:

In [8]:
# Compute only specific quantities of the periodic optics
#   "beta1" = horizontal-like beta
#   "phi2" = vertical-like phase advance,
#   "dx" = horizontal dispersion
tw = sb.twiss(fodo, cols=["beta1", "phi2", "dx"])
tw

Twiss:
summ:
 q1     = 0.05489859401259565
 q2     = 0.05401330001000861
 etac   = 0.019165878242797245
 alphac = 0.019165879048722943

df:
11×7 DataFrame
 Row │ index  name  kind        s        beta1     phi2        dx
     │                          [m]      [m]       [2π]        [m]
─────┼─────────────────────────────────────────────────────────────────
   1 │     1  qf    Quadrupole  0.0      13.4106   0.0         1.72652
   2 │     2  sf    Sextupole   0.5      13.4105   0.00785137  1.72681
   3 │     3  b     SBend       0.7      12.9252   0.0108852   1.69474
   4 │     4  d     Drift       1.9      10.3228   0.0268789   1.51609
   5 │     5  qd    Quadrupole  2.0      10.1299   0.0280573   1.50236
   6 │     5  qd    Quadrupole  2.16667   9.91907  0.0299805   1.48716
   7 │     5  qd    Quadrupole  2.33333   9.91903  0.0318842   1.48725
   8 │     6  sd    Sextupole   2.5      10.1298   0.0338073   1.50264
   9 │     7  b     SBend       2.7      10.5192   0.0361862   1.53034
 

To see all available columns that can be provided to `cols`, see the [`twiss`](twiss.md)
docstring.

### Nonlinear Twiss

Beyond the purely linear optics, `twiss` can also compute higher order quantities, simply by requesting the associated column and/or increasing the order of the underlying truncated power series map (a Taylor map, also called a DA map) used internally. All quantities are computed using SciBmad's Lie series normal form analysis package [`NonlinearNormalForm`](https://github.com/bmad-sim/NonlinearNormalForm.jl). 

`twiss` exposes two independent "orders" that can be set:

- `chrom`, the order to which the energy deviation `δ` is truncated, for computing higher
  order chromatic quantities
- `order`, the truncated order of all individual phase space variables, for computing
  amplitude-dependent tunes and resonance driving terms

The tunes and the beta and alpha functions are known to one order less in `δ` than the map, so
for example, setting `chrom=2` will compute the chromaticities (the tunes to first order in
`δ`), and if requested in `cols`
the Montague `w1`/`w2` functions, second-order dispersions `dx_2`/`dy_2`, chromatic beta
beat `dbeta1`/`dbeta2` (equivalent to writing `dbeta1_1`/`dbeta2_1`), etc. In fact, we can
request a chromatic derivative of _any_ scalar-valued quantity, using the notation
`d<quantity>_<order>`. By omitting the `_<order>`, order is assumed to be one. For example,
`dc11` is the first derivative w.r.t. δ of the coupling matrix component `c11`, and `dc11_2`
is the second derivative w.r.t. δ.

In [9]:
# Compute higher order chromatic quantities such as
# the chromaticity, Montague W functions ("w1", "w2"),
# and 2nd order dispersions ("dx_2", "dy_2") by setting
# the order of δ (chrom) equal to 2
tw = sb.twiss(fodo, cols=["w1", "w2", "dx_2", "dy_2"], chrom=2)
tw

Twiss:
summ:
 q1     = 0.05489859401259565 + 0.07905747898014202 δ
 q2     = 0.05401330001000861 - 0.008253153240321399 δ
 etac   = 0.019165878242797165 - 0.03645782324708619 δ
 alphac = 0.019165879048722932 - 0.036457825633970944 δ

df:
11×8 DataFrame
 Row │ index  name  kind        s        w1       w2       dx_2      dy_2
     │                          [m]      [1]      [1]
─────┼────────────────────────────────────────────────────────────────────────────
   1 │     1  qf    Quadrupole  0.0      3.54327  2.2888   -5.06473  -1.77727e-14
   2 │     2  sf    Sextupole   0.5      3.53053  2.32137  -4.89516  -1.86629e-14
   3 │     3  b     SBend       0.7      3.3431   2.52065  -4.74081  -1.91552e-14
   4 │     4  d     Drift       1.9      3.18849  2.61841  -4.27158  -1.98931e-14
   5 │     5  qd    Quadrupole  2.0      3.17452  2.6279   -4.23376  -1.99598e-14
   6 │     5  qd    Quadrupole  2.16667  3.16412  2.63122  -4.20777  -1.9934e-14
   7 │     5  qd    Quadrupole  2.33333  3.17

When computed to a higher order, amplitude- and energy-dependent quantities, such as the tunes
`q1`/`q2`, are returned not as plain numbers but as `AmplitudeDependentValue`s - Taylor series
in the actions `J₁`, `J₂` (the amplitudes of the oscillations in the two transverse modes) and
the energy deviation `δ`:

In [10]:
tw.q2

AmplitudeDependentValue:
 0.05401330001000861 - 0.008253153240321399 δ


Derivatives of an `AmplitudeDependentValue` can be extracted by indexing the quantity with the order
of the derivative in each
variable{pytext}` (in Python with a call, as Python has no keywords in indexing)`. For example, the
linear chromaticity in `y` is $\partial q_2/\partial\delta$:

In [11]:
# Get the y-chromaticity
chromy = tw.q2(delta=1)
chromy

-0.008253153240321399

For chromatic derivatives only, you can also use the shorthand `d<quantity>_<order>` notation:

In [12]:
chromy = tw.dq2  # or equivalently tw.dq2_1
chromy

-0.008253153240321399

Likewise, {jl}`tw.q2[delta=2]`{py}`tw.q2(delta=2)` (or `tw.dq2_2`) is $\partial^2
q_2/\partial\delta^2$. The same mechanism extends naturally to purely amplitude-dependent tune
shifts (once `order`
is raised high enough to resolve them), using the `J1` and `J2` keyword arguments to extract tune derivatives w.r.t. the action variables.

We can also extract any so-called resonance driving term/detune coefficient - monomials of the nonlinear effective Hamiltonian in linear action-angle variables. And, if the beam is
coasting (without synchrotron oscillations, e.g. without RF cavities), we can even take chromatic derivatives of these quantities as described before. These quantities are defined in [Resonance driving terms and detuning](twiss.md).

In [13]:
# Resonance driving terms h3000, h2100
tw = sb.twiss(fodo, cols=["h3000", "h2100"])
tw

Twiss:
summ:
 q1     = 0.05489859401259562 + 0.079057478980142 δ
 q2     = 0.05401330001000862 - 0.008253153240321175 δ
 etac   = 0.019165878242797245 - 0.03645782324708643 δ
 alphac = 0.019165879048723012 - 0.03645782563397118 δ

df:
11×6 DataFrame
 Row │ index  name  kind        s        h3000                   h2100
     │                          [m]
─────┼───────────────────────────────────────────────────────────────────────────────
   1 │     1  qf    Quadrupole  0.0        0.303294+0.205128im   0.664041+0.223308im
   2 │     2  sf    Sextupole   0.5        0.278905+0.237228im   0.655396+0.247536im
   3 │     3  b     SBend       0.7      -0.0492589-0.338686im    0.53745-0.428855im
   4 │     4  d     Drift       1.9       0.0595667-0.338563im    0.57978-0.372724im
   5 │     5  qd    Quadrupole  2.0       0.0694723-0.33667im    0.583397-0.367036im
   6 │     5  qd    Quadrupole  2.16667   0.0862015-0.332779im    0.58943-0.357268im
   7 │     5  qd    Quadrupole  2.33333    0.10

To compute these quantities, technically `order=2` is required. However, `twiss` will automatically resolve the internal orders given the requested columns if not explicitly provided.

### Spin Analysis

`twiss` can additionally analyze the spin dynamics by setting {jl}`spin=True`{py}`spin=True`. This
enables
computation of the invariant spin field (as a Taylor series in the phase space coordinates if requested) and the amplitude-dependent spin tune:

In [14]:
# Spin analysis (invariant spin field as Taylor series,
# amplitude-dependent spin tune).
tw = sb.twiss(fodo, cols=["nx", "ny", "nz"], spin=True,
              as_taylor_series=True, order=2)
n = [tw.nx[1], tw.ny[1], tw.nz[1]]  # ISF at first element
n

[TPS64{GTPSA.Dynamic}:
Descriptor(NV=6, MO=2)
 COEFFICIENT             ORDER   EXPONENTS
-1.1129841237292276E-15    1     1 0  0 0  0 0
-5.0864748721502509E-15    1     0 1  0 0  0 0
 2.8726619691324142E+00    1     0 0  1 0  0 0
 7.1359575554618138E-01    1     0 0  0 1  0 0
 1.0328113898759132E-14    1     0 0  0 0  0 1
-1.0638523663010506E-15    2     2 0  0 0  0 0
 2.5851026154268499E-14    2     1 1  0 0  0 0
-2.9060104586387736E-14    2     0 2  0 0  0 0
-9.8609301475078208E+00    2     1 0  1 0  0 0
 2.5979472292584767E+00    2     0 1  1 0  0 0
-3.9730370193302643E-15    2     0 0  2 0  0 0
 4.6570660257458503E+00    2     1 0  0 1  0 0
 5.5944457622851665E+01    2     0 1  0 1  0 0
 6.5012509432810885E-14    2     0 0  1 1  0 0
-1.4838597166719928E-13    2     0 0  0 2  0 0
-4.3408048192810965E-14    2     1 0  0 0  0 1
-2.5384882094871271E-15    2     0 1  0 0  0 1
-1.6861654806572169E+01    2     0 0  1 0  0 1
 1.6098131881371625E+01    2     0 0  0 1  0 1
-9.980742681622612

With {jl}`as_taylor_series=True`{py}`as_taylor_series=True`, the components of the ISF are returned
as full Taylor series in
the phase space variables $(x,p_x,y,p_y,z,\delta)$, rather than just returning $\hat{n}_0$ (the invariant spin field on the closed orbit). The spin tune, accessible as `tw.qspin`, is likewise an `AmplitudeDependentValue`:

In [15]:
tw.qspin

AmplitudeDependentValue:
 -0.3094612779434053 - 0.3094612776559473 δ - 4.0543440034887456e-10 J₁ + 28.127501690102292 J₂ - 2.9509532661683825e-10 δ²


As with the orbital tunes, individual derivatives, such as the spin tune's linear dependence on energy, or on its amplitude `J₂`, can be indexed.

### Optimization using Autodiff
Let's check the chromaticities of our FODO cell:

In [16]:
tw = sb.twiss(fodo, chrom=2)
print("tw.dq1 =", tw.dq1)
print("tw.dq2 =", tw.dq2)

tw.dq1 = 0.07905747898014202
tw.dq2 = -0.008253153240321399


We want to adjust the sextupoles `sf` and `sd` so that these values are both `+1`. To do this, we first form a `TwissExpr` of our Beamline - this allows us to form mathematical expressions for _any_ output of regular `twiss`, which we can then use in optimizations.

In [17]:
te = sb.TwissExpr(fodo)
te

TwissExpr of a Beamline with 8 elements

Because `TwissExpr` allows expressions for any output from `twiss`, the `cols` keyword argument does not need to, and cannot, be provided.

We can now form the two target expressions using our `TwissExpr`, specifying that we want the chromaticities `dq1` and `dq2` to be `+1` (with a `TwissExpr`,
`==` does not compare two numbers: it makes a target).

In [18]:
[te.dq1 == 1, te.dq2 == 1]

[Target: q1[delta=1] == 1.0, Target: q2[delta=1] == 1.0]

Lastly, we need to specify our "knobs" - the things we want to change to achieve our targets. This
is done using {jl}`@knob`{py}`sb.Knob` with the normal sextupole strengths `Kn2` of `sf` and `sd`:

In [19]:
[sb.Knob(sf, "Kn2"), sb.Knob(sd, "Kn2")]

[sf.Kn2, sd.Kn2]

Putting this all together into SciBmad's {jl}`match!`{py}`match` function, we can now run the
optimization:

In [20]:
te = sb.TwissExpr(fodo)
sol = sb.match(vary=[sb.Knob(sf, "Kn2"), sb.Knob(sd, "Kn2")],
               targets=[te.dq1 == 1, te.dq2 == 1])
sol

Match converged (all targets met): 1 iterations, 2 evaluations, 1.11 s
 Target       value  goal    met
 q1[delta=1]  1.0    == 1.0  yes
 q2[delta=1]  1.0    == 1.0  yes
 Knob    before  after         lower  upper
 sf.Kn2  1.2     13.50047492   -Inf   Inf
 sd.Kn2  -1.2    -15.20232969  -Inf   Inf


We see that the match has been achieved, and can verify it with a regular `twiss` call:

In [21]:
sb.twiss(fodo, chrom=2)

Twiss:
summ:
 q1     = 0.05489859401259572 + 0.9999999999999974 δ
 q2     = 0.05401330001000853 + 0.9999999999999989 δ
 etac   = 0.01916587824279715 - 0.6742571783629614 δ
 alphac = 0.01916587904872292 - 0.6742571807498461 δ

df:
11×26 DataFrame
 Row │ index  name  kind        s        beta1     phi1        dx       x             beta2    phi ⋯
     │                          [m]      [m]       [2π]        [m]      [m]           [m]      [2π ⋯
─────┼──────────────────────────────────────────────────────────────────────────────────────────────
   1 │     1  qf    Quadrupole  0.0      13.4106   0.0         1.72652  -1.66533e-17  10.2966  0.0 ⋯
   2 │     2  sf    Sextupole   0.5      13.4105   0.00584419  1.72681  -2.10134e-17  10.2966  0.0
   3 │     3  b     SBend       0.7      12.9252   0.00826202  1.69474  -2.23928e-17  10.6921  0.0
   4 │     4  d     Drift       1.9      10.3228   0.0248246   1.51609  -3.1225e-17   13.3816  0.0
   5 │     5  qd    Quadrupole  2.0      10.1299   0.

Under the hood, {jl}`match!`{py}`match` uses SciBmad's automatic differentiation and parametric
normal form to achieve an exact, and fast, match.

`TwissExpr` enables powerful accelerator lattice design and optimization workflows. Outputs of `TwissExpr` (such as the optics functions) can be passed into other `TwissExpr`s as initial conditions. For example, after matching arc FODO cells, one can pass the output values easily into another `TwissExpr` that is then used to match the arc FODO cells to a straight section. Derivatives of various quantities including w.r.t. knobs can also be matched through the same interface, for example as done in the Linear Optics from Closed Orbits (LOCO) algorithm. For all details, see the [Matching and Optimization](optimize.md) section of the documentation.

### Track

To do particle tracking, a single function `track` is provided, which accepts initial phase
space coordinates as an `n_particle x 6` matrix{pytext}` (a NumPy array, or a list of lists)` (the
coordinates are described on the [Track](track.md) page):

In [22]:
# Track a particle with [x, px, y, py, z, pz]
# = [1e-3, 0, 0, 0, 0, 1e-3] for n_turns=10
res = sb.track(fodo, v0=[[1e-3, 0, 0, 0, 0, 1e-3]], n_turns=10)

# Also do spin quaternion tracking:
res = sb.track(fodo, v0=[[1e-3, 0, 0, 0, 0, 1e-3]], n_turns=10, spin=True)
# Apply spin quaternion to horizontal initial spin direction (FAST):
s = sb.track_spin(res.q, [1, 0, 0])

# Track a bunch, give an array of size n_particles x 6:
n_particles = 10
res = sb.track(fodo, v0=np.random.rand(n_particles, 6)*1e-5, n_turns=10, spin=True)
res

TrackResult of 10 particles:
config:
 n_turns                         = 10
 save_every_n_turns              = 1
 in_body_coordinates             = false
 save_reference                  = false
 scalar_params                   = false
 rf_on                           = true
 ramp_particle_energy_without_rf = false
 ramp_update_each_particle       = false
 batch_start                     = 1
 verbose                         = false
 groupsize                       = 0
 use_cpu_multithreading          = false
 use_KA                          = false
 use_explicit_SIMD               = true

elapsed_time = 0.000321846 seconds

start = 1
turns = 0:1:10

s, index, name, kind     the saved locations: the end of the beamline (s = 4.0)
state                    particle state codes, e.g. state(turn=n, particle=i)
x, px, y, py, z, pz      the coordinates, e.g. x(turn=n, particle=i)
v                        all six coordinates, e.g. v(turn=n, particle=i)
q                        spin quaternions (

The particles are saved at the end of the beamline after every turn by default (turn 0 holds the initial coordinates), and are read by turn and particle:

In [23]:
print(res.x(turn=10))                # x of all particles after turn 10
print(res.x(turn=..., particle=1))   # x of particle 1 at turns 0 to 10

[ 2.26831072e-05 -1.84145349e-05 -2.14991397e-05 -6.45233889e-06
 -1.00137562e-05  2.20882299e-05 -1.43289588e-05 -1.40821343e-05
 -1.02413514e-05  1.68513646e-05]
[5.34075032e-06 1.14138277e-05 1.81126833e-05 2.46468505e-05
 3.02449979e-05 3.42464031e-05 3.61792597e-05 3.58163908e-05
 3.32017538e-05 2.86447299e-05 2.26831072e-05]


:::{only} python
Python has no keywords in indexing: where Julia writes `res.x[turn=10]`, Python calls the
column, `res.x(turn=10)`, and `...` stands for Julia's `:` there.
:::

To save them at other places, give the elements (or positions inside elements) with `at`, and read
them with the element first, as for the `twiss` results, e.g.
{jl}`res.x[qd, turn=10]`{py}`res.x(qd, turn=10)`. See the [Track](track.md) page for more details.

On the CPU, all tracking will automatically be parallelized using single instruction,
multiple data (SIMD) if your hardware supports it. CPU multithreading can also be enabled by
starting Julia with threads, and setting
{jl}`use_cpu_multithreading=True`{py}`use_cpu_multithreading=True`{pytext}` (Julia's threads are set with sb.init(threads=...) before the first use of SciBmad)`.
For more details,
see the [CPU Parallelization](#cpuparallel) section of the manual.

For GPU parallelized tracking, simply initialize your initial particle coordinates as a GPU
array. For example:


```python
gpu = sb.device("cuda")
n_particles = 100000  # 100,000 particles
v0 = gpu.asarray(np.random.rand(n_particles, 6) * 1e-5)
res = sb.track(fodo, v0=v0, n_turns=10, spin=True)
```

Apple Metal GPUs only support 32-bit floats, so on a Mac use `Float32` coordinates; all of
the tracking will then be done in 32-bit:


```python
gpu = sb.device("metal")
v0 = gpu.asarray(np.random.rand(n_particles, 6) * 1e-5, dtype=np.float32)
res = sb.track(fodo, v0=v0, n_turns=10, spin=True)
```

For more details, see the [GPU Parallelization](#gpuparallel) section of the manual.

## Where to Go Next

This guide only scratched the surface. The rest of the documentation covers:

- [Matching and Optimization](optimize.md): optimize tunes, beta functions, chromaticities,
  resonance driving terms and more, at any location, by varying any element parameter, with
  exact derivatives; or get the values and derivatives of any quantity computed by `twiss` for your own
  optimizer
- [Twiss](twiss.md): all the Twiss columns, reading them at elements and inside elements, Twiss
  of a part of a beamline and of open lines, resonance driving terms and detuning coefficients
- [Parametric Normal Form](parametric-nf.md): the results of `twiss`, such as the tunes, as Taylor series in
  magnet strengths, lengths or any other parameter
- [Track](track.md): saving particles anywhere, other species, 32-bit coordinates, CPU and GPU
  parallelization; with [Tracking Methods](tracking-methods.md) per element,
  [Collective Effects](collective.md), [time-dependent parameters](timedependent.md) and [batches](batch.md) of lattice parameters
- The [closed orbit](co.md), [dynamic aperture](dynamic-aperture.md) and
  [frequency map analysis](fma.md), all of which run on GPUs
- Lattices: [elements](element.md), [beamlines](beamline.md), [deferred
  expressions](defexpr.md) for interdependent parameters, and [definition tools](def-tools.md)
  such as phase trombones

To get the most use out of SciBmad, we recommend going through the documentation sequentially, starting with the next page [Defining a LatElement](element.md).